In [1]:
import pandas as pd
import numpy as np
import cupy as cp
import os
import warnings
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import train_test_split
from xgboost import XGBRFRegressor
from sklearn.metrics import mean_squared_error
import shap
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 统一配置
# ==========================================
INPUT_FILE = "P4_Cleaned_Dataset.csv"
OPTUNA_TRIALS = 30  # 可根据需要下调至 15 或 20 进一步提速
# ==========================================

def objective(trial, X_train_np, y_train_np):
    """
    加速版全局寻优目标函数：
    使用 80/20 单次划分代替 5 折交叉验证，使寻优速度提升 80%。
    """
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 200, step=50),
        'max_depth': trial.suggest_int('max_depth', 6, 12),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.2, 0.6),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'tree_method': 'gpu_hist',
        'random_state': 42,
        'n_jobs': -1
    }
    
    # 单次划分加速寻优
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_np, y_train_np, test_size=0.2, random_state=42
    )
    
    model = XGBRFRegressor(**params)
    model.fit(cp.array(X_tr), cp.array(y_tr))
    
    preds = cp.asnumpy(model.predict(cp.array(X_val)))
    mse = mean_squared_error(y_val, preds)
        
    return mse

def run_shap_analysis(input_file):
    if not os.path.exists(input_file):
        print(f"文件未找到: {input_file}。请先运行数据固化脚本 01_prepare_P4_data.py。")
        return
        
    df = pd.read_csv(input_file)
    meta_cols = ['Year', 'Zone', 'latitude', 'longitude', 'yield']
    feature_cols = [col for col in df.columns if col not in meta_cols]
    
    X = df[feature_cols]
    y = df['yield']
    
    X_np = X.values
    y_np = y.values
    
    print(f"\n{'='*60}")
    print(">>> 第一步：执行全局贝叶斯优化 (加速版)")
    print(f"{'='*60}")
    
    study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=42))
    func = lambda trial: objective(trial, X_np, y_np)
    study.optimize(func, n_trials=OPTUNA_TRIALS)
    
    best_params = study.best_params
    best_params.update({'tree_method': 'gpu_hist', 'random_state': 42, 'n_jobs': -1})
    print(f"全局最优参数已找到: {best_params}")
    
    print(f"\n{'='*60}")
    print(">>> 第二步：基于 100% 全量数据训练最终全局模型")
    print(f"{'='*60}")
    # 这一步极其重要：用最优参数重新看遍所有数据，确保 SHAP 解释无死角
    final_model = XGBRFRegressor(**best_params)
    final_model.fit(cp.array(X_np), cp.array(y_np))
    
    print(f"\n{'='*60}")
    print(">>> 第三步：计算 SHAP 值并导出全套底层矩阵数据")
    print(f"{'='*60}")
    
    # SHAP TreeExplainer 计算
    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X)
    
    # ---------------- 导出底层数据 ----------------
    print("正在导出底层数据...")
    
    # 1. 导出 SHAP 值矩阵
    shap_df = pd.DataFrame(shap_values, columns=feature_cols)
    shap_df.to_csv("SHAP_Values_Matrix.csv", index=False)
    
    # 2. 导出对应的特征值矩阵 (与 SHAP 矩阵严格按行对应)
    X.to_csv("SHAP_Feature_Matrix.csv", index=False)
    
    # 3. 导出包含元数据信息的完整比对表
    full_export_df = df[meta_cols].copy()
    full_export_df['Predicted_Yield'] = cp.asnumpy(final_model.predict(cp.array(X_np)))
    full_export_df.to_csv("SHAP_Metadata_and_Predictions.csv", index=False)
    
    # 4. 导出 Expected Value (基线常数)
    expected_value = explainer.expected_value
    if isinstance(expected_value, np.ndarray):
        expected_value = expected_value[0]
    with open("SHAP_Expected_Value.txt", "w") as f:
        f.write(f"Expected Value (Baseline Yield): {expected_value}\n")
    # ----------------------------------------------
    
    print("正在生成默认 SHAP 图表...")
    
    # 图 1: SHAP 特征重要性条形图 (Summary Plot - Bar)
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X, plot_type="bar", show=False, max_display=15)
    plt.title("Global Feature Importance (SHAP)")
    plt.tight_layout()
    plt.savefig("SHAP_Importance_Bar.pdf", dpi=300, bbox_inches='tight')
    plt.close()
    
    # 图 2: SHAP 蜂巢图 (Summary Plot - Dot)
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X, show=False, max_display=15)
    plt.title("SHAP Summary Plot (Impact on Yield)")
    plt.tight_layout()
    plt.savefig("SHAP_Summary_Dot.pdf", dpi=300, bbox_inches='tight')
    plt.close()
    
    print("\n分析彻底完成！已在当前目录生成以下文件：")
    print(" [数据] SHAP_Values_Matrix.csv")
    print(" [数据] SHAP_Feature_Matrix.csv")
    print(" [数据] SHAP_Metadata_and_Predictions.csv")
    print(" [数据] SHAP_Expected_Value.txt")
    print(" [图表] SHAP_Importance_Bar.pdf")
    print(" [图表] SHAP_Summary_Dot.pdf")

if __name__ == "__main__":
    run_shap_analysis(INPUT_FILE)


>>> 第一步：执行全局贝叶斯优化 (加速版)
全局最优参数已找到: {'n_estimators': 100, 'max_depth': 12, 'colsample_bynode': 0.5330978818706961, 'subsample': 0.8566188586056606, 'tree_method': 'gpu_hist', 'random_state': 42, 'n_jobs': -1}

>>> 第二步：基于 100% 全量数据训练最终全局模型

>>> 第三步：计算 SHAP 值并导出全套底层矩阵数据
正在导出底层数据...
正在生成默认 SHAP 图表...

分析彻底完成！已在当前目录生成以下文件：
 [数据] SHAP_Values_Matrix.csv
 [数据] SHAP_Feature_Matrix.csv
 [数据] SHAP_Metadata_and_Predictions.csv
 [数据] SHAP_Expected_Value.txt
 [图表] SHAP_Importance_Bar.pdf
 [图表] SHAP_Summary_Dot.pdf
